# Analysis Zone

Objetivo de la fase:

1. Consumir exclusivamente `exploitation_zone.duckdb`.
2. Ejecutar visualizacion, clustering y prediccion.
3. Conservar los resultados como outputs del notebook, sin generar JSON, CSV, PNG ni modelos externos.


In [1]:
import math
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Iterable

import duckdb
import matplotlib.pyplot as plt

ROOT = Path.cwd()
EXPLOITATION_DB = ROOT / "exploitation_zone" / "exploitation_zone.duckdb"


def ensure_phase_dirs() -> None:
    pass


In [2]:
def read_duckdb_rows(db_path: Path, table_name: str, order_by: str | None = None) -> list[dict[str, object]]:
    if duckdb is None:
        raise RuntimeError("DuckDB no está instalado.")
    conn = duckdb.connect(str(db_path), read_only=True)
    try:
        query = f'SELECT * FROM "{table_name}"'
        if order_by:
            query += f" ORDER BY {order_by}"
        cursor = conn.execute(query)
        columns = [description[0] for description in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]
    finally:
        conn.close()

def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0

def euclidean_distance(a: list[float], b: list[float]) -> float:
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

def standardize_matrix(matrix: list[list[float]]) -> tuple[list[list[float]], list[float], list[float]]:
    columns = list(zip(*matrix))
    means = [mean(list(col)) for col in columns]
    stds = []
    for idx, col in enumerate(columns):
        variance = mean([(value - means[idx]) ** 2 for value in col])
        stds.append(math.sqrt(variance) if variance > 0 else 1.0)
    scaled = []
    for row in matrix:
        scaled.append([(value - means[idx]) / stds[idx] for idx, value in enumerate(row)])
    return scaled, means, stds

def run_kmeans(matrix: list[list[float]], k: int = 4, max_iter: int = 40) -> tuple[list[int], list[list[float]], float]:
    if not matrix:
        return [], [], 0.0
    k = max(1, min(k, len(matrix)))
    seed_indexes = sorted({int(i * len(matrix) / k) for i in range(k)})
    centroids = [matrix[idx][:] for idx in seed_indexes]
    labels = [0] * len(matrix)
    for _ in range(max_iter):
        new_labels = []
        for row in matrix:
            distances = [euclidean_distance(row, centroid) for centroid in centroids]
            new_labels.append(min(range(len(distances)), key=distances.__getitem__))
        if new_labels == labels:
            break
        labels = new_labels
        for cluster_id in range(len(centroids)):
            cluster_rows = [row for row, label in zip(matrix, labels) if label == cluster_id]
            if not cluster_rows:
                continue
            centroids[cluster_id] = [
                mean([row[col_idx] for row in cluster_rows])
                for col_idx in range(len(cluster_rows[0]))
            ]
    inertia = 0.0
    for row, label in zip(matrix, labels):
        inertia += euclidean_distance(row, centroids[label]) ** 2
    return labels, centroids, round(inertia, 6)

def mae(y_true: list[float], y_pred: list[float]) -> float:
    return mean([abs(y - y_hat) for y, y_hat in zip(y_true, y_pred)])

def rmse(y_true: list[float], y_pred: list[float]) -> float:
    return math.sqrt(mean([(y - y_hat) ** 2 for y, y_hat in zip(y_true, y_pred)]))

def r2_score(y_true: list[float], y_pred: list[float]) -> float:
    y_mean = mean(y_true)
    ss_tot = sum((y - y_mean) ** 2 for y in y_true)
    ss_res = sum((y - y_hat) ** 2 for y, y_hat in zip(y_true, y_pred))
    return 0.0 if ss_tot == 0 else 1 - (ss_res / ss_tot)

def dot_product(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))

def train_linear_regression_gradient_descent(
    x_train: list[list[float]],
    y_train: list[float],
    learning_rate: float = 0.03,
    epochs: int = 1200,
) -> list[float]:
    if not x_train:
        return []
    weights = [0.0] * (len(x_train[0]) + 1)
    sample_count = len(x_train)
    for _ in range(epochs):
        gradients = [0.0] * len(weights)
        for row, target in zip(x_train, y_train):
            prediction = weights[0] + dot_product(weights[1:], row)
            error = prediction - target
            gradients[0] += error
            for idx, value in enumerate(row, start=1):
                gradients[idx] += error * value
        for idx in range(len(weights)):
            weights[idx] -= learning_rate * (gradients[idx] / sample_count)
    return weights

def predict_linear_regression(weights: list[float], x_rows: list[list[float]]) -> list[float]:
    if not weights:
        return []
    return [weights[0] + dot_product(weights[1:], row) for row in x_rows]

def knn_predict(
    x_train: list[list[float]],
    y_train: list[float],
    x_test: list[list[float]],
    k: int = 7,
) -> list[float]:
    if not x_train or not x_test:
        return []
    effective_k = max(1, min(k, len(x_train)))
    predictions = []
    for row in x_test:
        distances = [(euclidean_distance(row, train_row), target) for train_row, target in zip(x_train, y_train)]
        neighbors = sorted(distances, key=lambda item: item[0])[:effective_k]
        predictions.append(mean([target for _, target in neighbors]))
    return predictions

def add_calendar_features(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    enriched = []
    for row in rows:
        date_value = datetime.fromisoformat(str(row["date"]))
        enriched_row = dict(row)
        enriched_row["month"] = date_value.month
        enriched_row["day_of_week"] = date_value.weekday()
        enriched_row["is_weekend"] = 1 if date_value.weekday() >= 5 else 0
        enriched.append(enriched_row)
    return enriched

def save_plot(title: str, x_values: list[object], y_values: list[float], kind: str = "bar") -> str:
    plt.figure(figsize=(10, 5))
    if kind == "line":
        plt.plot(x_values, y_values, marker="o", linewidth=1.5)
    else:
        plt.bar(x_values, y_values)
        plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return title


In [3]:
def describe_clusters(rows: list[dict[str, object]], labels: list[int]) -> list[dict[str, object]]:
    grouped: dict[int, list[dict[str, object]]] = defaultdict(list)
    for row, label in zip(rows, labels):
        grouped[label].append(row)
    summaries = []
    for label in sorted(grouped):
        cluster_rows = grouped[label]
        summaries.append(
            {
                "cluster_id": label,
                "zone_count": len(cluster_rows),
                "districts": sorted({str(row["district_name"]) for row in cluster_rows}),
                "avg_price": round(mean([float(row["avg_price"]) for row in cluster_rows]), 4),
                "median_price": round(mean([float(row["median_price"]) for row in cluster_rows]), 4),
                "avg_rating": round(mean([float(row["avg_rating"]) for row in cluster_rows]), 4),
                "avg_accommodates": round(mean([float(row["avg_accommodates"]) for row in cluster_rows]), 4),
                "avg_tourism_asset_score": round(mean([float(row["neighborhood_tourism_asset_score"]) for row in cluster_rows]), 4),
                "avg_listing_count": round(mean([float(row["listing_count"]) for row in cluster_rows]), 4),
            }
        )
    return summaries

def run_visualization_pipeline() -> dict[str, object]:
    airbnb_zones = sorted(read_duckdb_rows(EXPLOITATION_DB, "airbnb_zone_features"), key=lambda row: row["avg_price"], reverse=True)
    weather_daily = read_duckdb_rows(EXPLOITATION_DB, "weather_daily", order_by="date")
    top_zones = airbnb_zones[:10]
    summary_rows = [
        {
            "district_name": row["district_name"],
            "neighborhood_name": row["neighborhood_name"],
            "listing_count": row["listing_count"],
            "avg_price": row["avg_price"],
            "neighborhood_tourism_asset_score": row["neighborhood_tourism_asset_score"],
        }
        for row in top_zones
    ]
    zone_plot = save_plot(
        "Precio medio Airbnb por zona",
        [f"{row['district_name']} / {row['neighborhood_name']}" for row in top_zones],
        [row["avg_price"] for row in top_zones],
    )
    weather_plot = save_plot(
        "Temperatura media diaria",
        [row["date"] for row in weather_daily],
        [row["temperature_avg"] for row in weather_daily],
        kind="line",
    )
    result = {"top_districts": summary_rows, "visualizations": [plot for plot in [zone_plot, weather_plot] if plot]}
    return result


In [4]:
def run_clustering_pipeline() -> dict[str, object]:
    zone_rows = [row for row in read_duckdb_rows(EXPLOITATION_DB, "airbnb_zone_features") if row["listing_count"] >= 15]
    feature_names = [
        "avg_price",
        "median_price",
        "avg_rating",
        "avg_accommodates",
        "avg_bedrooms",
        "avg_availability_365",
        "entire_home_ratio",
        "private_room_ratio",
        "superhost_ratio",
        "instant_bookable_ratio",
        "avg_review_count",
        "neighborhood_tourism_asset_score",
        "district_tourism_asset_score",
    ]
    matrix = [[float(row[name]) for name in feature_names] for row in zone_rows]
    scaled, means, stds = standardize_matrix(matrix)
    labels, _, inertia = run_kmeans(scaled, k=4)
    enriched_rows = []
    for row, label in zip(zone_rows, labels):
        enriched = dict(row)
        enriched["cluster_id"] = label
        enriched_rows.append(enriched)
    cluster_summary = describe_clusters(zone_rows, labels)
    cluster_plot = save_plot(
        "Precio medio por zona clusterizada",
        [f"{row['district_name']} / {row['neighborhood_name']}" for row in enriched_rows[:20]],
        [row["avg_price"] for row in enriched_rows[:20]],
    )
    return {"zones_clustered": len(enriched_rows), "k": 4, "inertia": inertia, "feature_names": feature_names, "cluster_summary": cluster_summary, "visualizations": [plot for plot in [cluster_plot] if plot]}


In [5]:
def run_prediction_pipeline() -> dict[str, object]:
    zone_day_rows = add_calendar_features(read_duckdb_rows(EXPLOITATION_DB, "airbnb_zone_day_features", order_by="date, district_name, neighborhood_name"))
    zone_day_rows = [row for row in zone_day_rows if row["listing_count"] >= 15]
    feature_names = [
        "avg_calendar_price",
        "avg_minimum_nights",
        "listing_count",
        "neighborhood_tourism_asset_score",
        "district_tourism_asset_score",
        "temperature_avg",
        "humidity_avg",
        "radiation_avg",
        "wind_speed_avg",
        "rain_total",
        "month",
        "day_of_week",
        "is_weekend",
    ]
    matrix = [[float(row[name]) for name in feature_names] for row in zone_day_rows]
    targets = [float(row["booked_rate"]) for row in zone_day_rows]
    scaled_matrix, means, stds = standardize_matrix(matrix)
    split = max(1, int(len(scaled_matrix) * 0.8))
    x_train, x_test = scaled_matrix[:split], scaled_matrix[split:] or scaled_matrix[:]
    y_train, y_test = targets[:split], targets[split:] or targets[:]
    test_rows = zone_day_rows[split:] or zone_day_rows[:]

    baseline_value = mean(y_train)
    baseline_predictions = [baseline_value] * len(x_test)
    linear_weights = train_linear_regression_gradient_descent(x_train, y_train)
    linear_predictions = [min(1.0, max(0.0, value)) for value in predict_linear_regression(linear_weights, x_test)]
    knn_k = max(3, min(15, int(len(x_train) ** 0.5)))
    knn_predictions = [min(1.0, max(0.0, value)) for value in knn_predict(x_train, y_train, x_test, k=knn_k)]

    model_metrics = [
        {"model": "baseline_mean", "rmse": round(rmse(y_test, baseline_predictions), 6), "mae": round(mae(y_test, baseline_predictions), 6), "r2": round(r2_score(y_test, baseline_predictions), 6)},
        {"model": "linear_regression", "rmse": round(rmse(y_test, linear_predictions), 6), "mae": round(mae(y_test, linear_predictions), 6), "r2": round(r2_score(y_test, linear_predictions), 6)},
        {"model": f"knn_regression_k_{knn_k}", "rmse": round(rmse(y_test, knn_predictions), 6), "mae": round(mae(y_test, knn_predictions), 6), "r2": round(r2_score(y_test, knn_predictions), 6)},
    ]
    best_model = min(model_metrics, key=lambda item: item["rmse"])
    prediction_rows = []
    for index, row in enumerate(test_rows):
        prediction_rows.append(
            {
                "date": row["date"],
                "district_name": row["district_name"],
                "neighborhood_name": row["neighborhood_name"],
                "booked_rate_observed": round(y_test[index], 6),
                "booked_rate_baseline_mean": round(baseline_predictions[index], 6),
                "booked_rate_linear_regression": round(linear_predictions[index], 6),
                f"booked_rate_knn_regression_k_{knn_k}": round(knn_predictions[index], 6),
                "avg_calendar_price": row["avg_calendar_price"],
                "listing_count": row["listing_count"],
                "temperature_avg": row["temperature_avg"],
                "rain_total": row["rain_total"],
            }
        )
    metrics = {
        "target": "booked_rate",
        "train_rows": len(x_train),
        "test_rows": len(x_test),
        "models": model_metrics,
        "best_model": best_model["model"],
        "feature_names": feature_names,
        "standardization_means": means,
        "standardization_stds": stds,
        "linear_regression_weights": linear_weights,
        "knn_k": knn_k,
        "source_table": "airbnb_zone_day_features",
        "source_database": str(EXPLOITATION_DB.relative_to(ROOT)),
    }
    rmse_plot = save_plot("Comparativa RMSE de modelos", [item["model"] for item in model_metrics], [item["rmse"] for item in model_metrics])
    best_model_plot = save_plot(
        f"Booked rate observado vs predicho ({best_model['model']})",
        [f"{row['district_name']} / {row['neighborhood_name']}" for row in prediction_rows[:20]],
        [row["booked_rate_observed"] for row in prediction_rows[:20]],
    )
    return {"target": "booked_rate", "source_table": "airbnb_zone_day_features", "train_rows": len(x_train), "test_rows": len(x_test), "models": model_metrics, "best_model": best_model["model"], "feature_names": feature_names, "visualizations": [plot for plot in [rmse_plot, best_model_plot] if plot], "prediction_sample": prediction_rows[:20]}


In [6]:
ensure_phase_dirs()
analysis_results = {
    "visualization": run_visualization_pipeline(),
    "clustering": run_clustering_pipeline(),
    "prediction": run_prediction_pipeline(),
    "exploitation_database": {"path": str(EXPLOITATION_DB.relative_to(ROOT))},
}
analysis_results


{'visualization': {'top_zones_preview': [{'district_name': 'Gràcia', 'neighborhood_name': 'el Coll', 'listing_count': 35, 'avg_price': 403.4286}, {'district_name': 'Sant Andreu', 'neighborhood_name': 'Navas', 'listing_count': 54, 'avg_price': 342.463}, {'district_name': 'Sants-Montjuïc', 'neighborhood_name': 'Sants - Badal', 'listing_count': 156, 'avg_price': 328.4615}], 'visualizations': ['Precio medio Airbnb por zona', 'Temperatura media diaria']}, 'clustering': {'zones_clustered': 55, 'k': 4, 'inertia': 426.664308, 'visualizations': ['Precio medio por zona clusterizada']}, 'prediction': {'target': 'booked_rate', 'source_table': 'airbnb_zone_day_features', 'train_rows': 16078, 'test_rows': 4020, 'models': [{'model': 'baseline_mean', 'rmse': 0.11591, 'mae': 0.092797, 'r2': -0.235243}, {'model': 'linear_regression', 'rmse': 0.133295, 'mae': 0.111635, 'r2': -0.633581}, {'model': 'knn_regression_k_15', 'rmse': 0.068605, 'mae': 0.054031, 'r2': 0.567266}], 'best_model': 'knn_regression_k_1